In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision.transforms as transforms

from torchvision import datasets
from torchvision.models import resnet18, ResNet18_Weights

from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from tqdm import tqdm

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("="*40)
print("Device :", device)

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

print("="*40)

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
TRAIN_DIR = "D:\\BrainTumorNet\\Dataset\\Testing"

TEST_DIR = "D:\\BrainTumorNet\\Dataset\\Training"

In [ ]:
IMAGE_SIZE = 224

BATCH_SIZE = 32

LEARNING_RATE = 1e-4

EPOCHS = 15

NUM_WORKERS = 4

NUM_CLASSES = 4

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=test_transform
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

In [ ]:
print(train_dataset.classes)

print(train_dataset.class_to_idx)

In [ ]:
images, labels = next(iter(train_loader))

plt.figure(figsize=(12,8))

for i in range(8):

    img = images[i].permute(1,2,0)

    img = img * torch.tensor([0.229,0.224,0.225]) + \
          torch.tensor([0.485,0.456,0.406])

    img = img.numpy()

    img = np.clip(img,0,1)

    plt.subplot(2,4,i+1)

    plt.imshow(img)

    plt.title(train_dataset.classes[labels[i]])

    plt.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
weights = ResNet18_Weights.DEFAULT

model = resnet18(weights=weights)

print("Pretrained ResNet18 Loaded Successfully!")

In [ ]:
print(model)

In [ ]:
for param in model.parameters():
    param.requires_grad = False

In [ ]:
model.fc = nn.Sequential(

    nn.Linear(model.fc.in_features, 256),

    nn.ReLU(),

    nn.Dropout(0.5),

    nn.Linear(256, NUM_CLASSES)

)

In [ ]:
model = model.to(device)

print(model.fc)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total Parameters     : {total_params:,}")

print(f"Trainable Parameters : {trainable_params:,}")

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer = torch.optim.Adam(

    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),

    lr=LEARNING_RATE

)

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode="max",

    factor=0.5,

    patience=2,

    verbose=True

)

In [ ]:
images, labels = next(iter(train_loader))

images = images.to(device)

outputs = model(images)

print("Output Shape :", outputs.shape)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in tqdm(loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = outputs.max(1)

        total += labels.size(0)

        running_correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(loader)

    epoch_acc = running_correct / total

    return epoch_loss, epoch_acc

In [ ]:
def validate_one_epoch(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    running_correct = 0
    total = 0

    predictions = []
    ground_truth = []

    with torch.no_grad():

        for images, labels in tqdm(loader):

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = outputs.max(1)

            total += labels.size(0)

            running_correct += predicted.eq(labels).sum().item()

            predictions.extend(predicted.cpu().numpy())
            ground_truth.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader)

    epoch_acc = running_correct / total

    return epoch_loss, epoch_acc, predictions, ground_truth

In [ ]:
train_losses = []

valid_losses = []

train_accuracies = []

valid_accuracies = []

In [ ]:
best_accuracy = 0

In [ ]:
for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    valid_loss, valid_acc, predictions, ground_truth = validate_one_epoch(
        model,
        test_loader,
        criterion,
        device
    )

    scheduler.step(valid_acc)

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    train_accuracies.append(train_acc)
    valid_accuracies.append(valid_acc)

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Accuracy : {train_acc:.4f}")

    print(f"Validation Loss : {valid_loss:.4f}")
    print(f"Validation Accuracy : {valid_acc:.4f}")

    if valid_acc > best_accuracy:

        best_accuracy = valid_acc

        torch.save(model.state_dict(), "resnet18_best.pth")

        print("✅ Best Model Saved")

In [ ]:
print(f"Best Validation Accuracy : {best_accuracy:.4f}")

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(train_losses, label="Train Loss")
plt.plot(valid_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training & Validation Loss")

plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(train_accuracies, label="Train Accuracy")
plt.plot(valid_accuracies, label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training & Validation Accuracy")

plt.legend()

plt.show()

In [ ]:
print(classification_report(
    ground_truth,
    predictions,
    target_names=train_dataset.classes
))

In [ ]:
cm = confusion_matrix(
    ground_truth,
    predictions
)

print(cm)

In [ ]:
best_model = resnet18(weights=None)

best_model.fc = nn.Sequential(
    nn.Linear(best_model.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, NUM_CLASSES)
)

best_model.load_state_dict(torch.load("resnet18_best.pth"))

best_model = best_model.to(device)

best_model.eval()

print("Best Model Loaded Successfully!")

In [ ]:
images, labels = next(iter(test_loader))

images = images.to(device)

with torch.no_grad():

    outputs = best_model(images)

    _, predictions = torch.max(outputs, 1)

In [ ]:
plt.figure(figsize=(15,10))

for i in range(8):

    image = images[i].cpu()

    image = image.permute(1,2,0)

    image = image * torch.tensor([0.229,0.224,0.225]) + \
            torch.tensor([0.485,0.456,0.406])

    image = image.numpy()

    image = np.clip(image,0,1)

    plt.subplot(2,4,i+1)

    plt.imshow(image)

    plt.title(
        f"Pred : {train_dataset.classes[predictions[i]]}\n"
        f"True : {train_dataset.classes[labels[i]]}"
    )

    plt.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
accuracy = accuracy_score(
    ground_truth,
    predictions.cpu().numpy()
)

precision = precision_score(
    ground_truth,
    predictions.cpu().numpy(),
    average="weighted"
)

recall = recall_score(
    ground_truth,
    predictions.cpu().numpy(),
    average="weighted"
)

f1 = f1_score(
    ground_truth,
    predictions.cpu().numpy(),
    average="weighted"
)

In [ ]:
print("="*40)

print(f"Accuracy  : {accuracy:.4f}")

print(f"Precision : {precision:.4f}")

print(f"Recall    : {recall:.4f}")

print(f"F1 Score  : {f1:.4f}")

print("="*40)

In [ ]:
plt.figure(figsize=(8,6))

plt.imshow(cm, cmap="Blues")

plt.title("Confusion Matrix")

plt.colorbar()

plt.xticks(
    range(NUM_CLASSES),
    train_dataset.classes,
    rotation=45
)

plt.yticks(
    range(NUM_CLASSES),
    train_dataset.classes
)

for i in range(NUM_CLASSES):

    for j in range(NUM_CLASSES):

        plt.text(
            j,
            i,
            cm[i,j],
            ha="center",
            va="center",
            color="red"
        )

plt.xlabel("Predicted")

plt.ylabel("True")

plt.tight_layout()

plt.show()

In [ ]:
results = pd.DataFrame({

    "Metric":[
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],

    "Value":[
        accuracy,
        precision,
        recall,
        f1
    ]

})

results

In [ ]:
results.to_csv(
    "resnet18_results.csv",
    index=False
)

print("Results Saved Successfully!")

In [ ]:
torch.save({

    "model_state_dict":best_model.state_dict(),

    "optimizer_state_dict":optimizer.state_dict(),

    "accuracy":accuracy

},"resnet18_checkpoint.pth")

print("Checkpoint Saved!")

In [ ]:
print("="*60)

print("BrainVision - ResNet18 Transfer Learning")

print("="*60)

print(f"Training Images : {len(train_dataset)}")

print(f"Testing Images  : {len(test_dataset)}")

print(f"Classes         : {train_dataset.classes}")

print(f"Best Accuracy   : {accuracy:.4f}")

print("="*60)

print("Project Completed Successfully!")

print("="*60)